# ETL · E-Commerce (2021-2025)

**Objetivo:** convertir los datos crudos de una tienda online en un conjunto de tablas limpias, coherentes y validadas, listas para analizarlas en Power BI.

**ETL** son las siglas de *Extract, Transform, Load* (extraer, transformar y cargar). Son los tres bloques de este notebook:

| Bloque | Qué hacemos | Secciones |
|---|---|---|
| **Extracción** | Leer los ficheros originales sin modificarlos | 1 |
| **Exploración** | Entender qué contiene cada tabla y detectar problemas *antes* de tocar nada | 2 |
| **Transformación** | Corregir y reorganizar los datos según lo encontrado | 3 |
| **Validación** | Comprobar que el resultado cuadra con las cifras oficiales del dataset | 4 |
| **Carga** | Exportar las tablas limpias en CSV | 5 |

**Entrada:** los 5 ficheros de `0. Datos crudos/E-Comerce/`.
**Salida:** 5 tablas en `1. ETL E-comerce/Datos_limpios/` organizadas como un **modelo en estrella** (se explica en la sección 3).

> Cada sección empieza con una nota que explica **qué** se hace y **por qué**, de modo que el notebook pueda seguirse sin conocer el proyecto. Se recomienda ejecutarlo de arriba abajo (*Run All*).

## 0. Preparación del entorno

Usamos tres librerías:

- **pandas**: trabaja con tablas (llamadas *DataFrames*). Es la herramienta principal del ETL.
- **numpy**: cálculos numéricos; pandas se apoya en ella.
- **pathlib**: construye rutas de carpetas que funcionan igual en Windows, Mac y Linux.

> **Si aparece `ModuleNotFoundError: No module named 'pandas'`**, el *kernel* (el Python que ejecuta el notebook) no tiene pandas instalado. Hay dos soluciones: elegir otro kernel en *Select Kernel* (arriba a la derecha), o quitar el `#` de la línea `%pip install` de la celda siguiente, ejecutarla una vez y reiniciar el kernel.

In [1]:
# %pip install pandas numpy        # <- descomentar solo si falta pandas (ver nota anterior)

import pandas as pd
import numpy as np
from pathlib import Path

# Opciones de visualización: ver todas las columnas y números con 2 decimales
pd.options.display.max_columns = 60
pd.options.display.float_format = "{:,.2f}".format

## 1. Extracción: carga de los archivos

Leemos los datos directamente de la carpeta `0. Datos crudos`. Así hay **una única fuente original** que nunca se modifica: si algo sale mal, basta con volver a ejecutar el notebook.

La ruta es *relativa* a la ubicación del notebook (`..` significa "subir una carpeta", hasta `6. Proyecto final`).

| Archivo | Variable | Qué representa **una fila** |
|---|---|---|
| `ecommerce_sales_customer_analytics_150k.csv` | `ventas` | Un pedido (cabecera: fecha, cliente, pago, envío, totales...) |
| `order_items.csv` | `items` | Un producto dentro de un pedido (línea de pedido) |
| `customer_master.csv` | `clientes` | Un cliente registrado |
| `product_catalog.csv` | `productos` | Un producto del catálogo |
| `dataset_statistics.csv` | `estadisticas` | Una sola fila con los totales oficiales del dataset (la usaremos para validar) |

Con `parse_dates` le pedimos a pandas que lea `order_date` como **fecha** y no como texto; así luego podremos extraer año, mes, día de la semana, etc.

In [2]:
RUTA_CRUDOS = Path("../0. Datos crudos/E-Comerce")

ventas       = pd.read_csv(RUTA_CRUDOS / "ecommerce_sales_customer_analytics_150k.csv", parse_dates=["order_date"])
items        = pd.read_csv(RUTA_CRUDOS / "order_items.csv")
clientes     = pd.read_csv(RUTA_CRUDOS / "customer_master.csv")
productos    = pd.read_csv(RUTA_CRUDOS / "product_catalog.csv")
estadisticas = pd.read_csv(RUTA_CRUDOS / "dataset_statistics.csv")

# Diccionario con las 4 tablas principales: nos permite recorrerlas con un bucle
tablas = {"ventas": ventas, "items": items, "clientes": clientes, "productos": productos}

for nombre, df in tablas.items():
    print(f"{nombre:<10} {df.shape[0]:>8,} filas x {df.shape[1]:>2} columnas")

ventas      138,116 filas x 46 columnas
items       397,569 filas x 12 columnas
clientes     25,000 filas x 11 columnas
productos     1,175 filas x  9 columnas


## 2. Exploración: entender los datos antes de transformarlos

Antes de cambiar nada hay que saber **qué hay** en cada tabla. Las decisiones de la sección 3 se basan en lo que encontremos aquí.

### 2.1 Primer vistazo a cada tabla

Mostramos las 3 primeras filas de cada tabla para ver qué columnas tiene y qué aspecto tienen los valores.

In [3]:
for nombre, df in tablas.items():
    print(f"=== {nombre} ===")
    display(df.head(3))

=== ventas ===


,order_id,order_date,order_time,order_status,sales_channel,customer_id,customer_name,customer_age,gender,customer_segment,customer_type,customer_city,customer_state,customer_country,region,customer_postal_code,payment_method,payment_status,currency,shipping_method,warehouse,delivery_days,estimated_delivery_days,delivery_status,return_status,return_reason,customer_rating,review_sentiment,customer_review,marketing_channel,campaign_name,coupon_code,loyalty_points_earned,loyalty_points_redeemed,quantity,gross_sales,discount_amount,tax_amount,shipping_cost,net_sales,product_cost,profit,profit_margin_percentage,customer_lifetime_value,is_repeat_customer,customer_order_count
0,ORD-301242,2023-11-06,16:37:47,Completed,Mobile App,CUST-003102,Jasmine Ryan,55,Male,Consumer,Loyal,Lake Williamberg,Texas,USA,South,86040,Digital Wallet,Paid,USD,Standard,WH-003,3.00,3.00,On Time,NaN,NaN,3.50,Positive,Satisfied with the purchase.,Direct,Default_Campaign,NaN,94,44,5,"1,350.19",477.89,61.07,12.03,945.40,588.33,345.04,36.50,"12,459.68",True,11
1,ORD-773460,2025-12-24,01:22:36,Completed,Website,CUST-003124,Scott Chase,26,Male,Premium,Loyal,Kristyport,Baden-Württemberg,Germany,South,62538,Debit Card,Pending,EUR,Economy,WH-005,10.00,10.00,On Time,NaN,NaN,3.60,Positive,Good value for money.,Direct,Default_Campaign,NaN,201,171,8,"3,144.64","1,456.06",320.83,9.00,"2,018.41","2,031.88",-22.47,-1.11,"13,032.48",True,11
2,ORD-449374,2021-07-05,14:24:21,Completed,Mobile App,CUST-012496,Marc Wheeler,69,Female,Premium,Loyal,Singletonhaven,New York,USA,East,19993,Cash on Delivery,Paid,USD,Express,WH-015,3.00,3.00,On Time,NaN,NaN,4.30,Positive,Good product. Works as expected.,YouTube,NaN,NaN,46,16,5,522.81,97.30,29.79,13.05,468.35,257.73,197.57,42.18,"6,159.44",True,6


=== items ===


,order_id,product_id,quantity,unit_price,discount_percentage,discount_amount,gross_sales,tax_amount,shipping_cost,net_sales,product_cost,profit
0,ORD-301242,PROD-000738,2,244.40,0.34,166.00,488.80,22.60,6.48,351.88,173.28,172.12
1,ORD-301242,PROD-000181,3,287.13,0.36,311.89,861.39,38.47,5.55,593.52,415.05,172.92
2,ORD-773460,PROD-000973,4,754.80,0.47,"1,413.89","3,019.20",305.01,9.00,"1,919.32","1,976.16",-65.84


=== clientes ===


,customer_id,customer_name,customer_age,gender,customer_segment,customer_city,customer_state,customer_country,region,customer_postal_code,customer_acquisition_cost
0,CUST-000001,Donna Miller,54,Female,Consumer,Laurenland,Dubai,UAE,South,11253,13.58
1,CUST-000002,Joseph James,61,Male,Consumer,Lake Peter,Lower Saxony,Germany,North,47778,27.45
2,CUST-000003,Daniel Smith,47,Male,Consumer,Seanview,North Rhine,Germany,West,13900,31.53


=== productos ===


,product_id,product_name,product_category,product_subcategory,brand,supplier,unit_price,product_cost,product_rating
0,PROD-000001,Apple Sharable bifurcated algorithm Smartphones,Electronics,Smartphones,Apple,Alibaba,686.90,474.06,4.30
1,PROD-000002,Google User-centric even-keeled encryption Sma...,Electronics,Smartphones,Google,Euro Logistics,978.26,519.66,2.90
2,PROD-000003,LG Face-to-face client-driven support Smartphones,Electronics,Smartphones,LG,Asian Manufacturing,275.51,185.48,3.90


**Qué contiene la tabla `ventas` (46 columnas), agrupada por temas:**

| Tema | Columnas | Comentario |
|---|---|---|
| Pedido | `order_id`, `order_date`, `order_time`, `order_status`, `sales_channel` | `order_id` identifica el pedido |
| Cliente | `customer_id`, `customer_name`, `customer_age`, `gender`, `customer_segment`, `customer_type`, `customer_city`, `customer_state`, `customer_country`, `region`, `customer_postal_code` | Se repiten en cada pedido del mismo cliente (lo comprobamos en 2.7) |
| Pago | `payment_method`, `payment_status`, `currency` | |
| Envío | `shipping_method`, `warehouse`, `delivery_days`, `estimated_delivery_days`, `delivery_status` | |
| Devolución y opinión | `return_status`, `return_reason`, `customer_rating`, `review_sentiment`, `customer_review` | |
| Marketing | `marketing_channel`, `campaign_name`, `coupon_code`, `loyalty_points_earned`, `loyalty_points_redeemed` | |
| Importes | `quantity`, `gross_sales`, `discount_amount`, `tax_amount`, `shipping_cost`, `net_sales`, `product_cost`, `profit`, `profit_margin_percentage` | Su relación se comprueba en 2.6 |
| Métricas de cliente | `customer_lifetime_value`, `is_repeat_customer`, `customer_order_count` | Valores precalculados por la fuente |

Las otras tablas son más sencillas: `items` tiene los importes de cada producto dentro del pedido, `clientes` los datos personales y geográficos, y `productos` la categoría, marca, proveedor, precio y coste de cada artículo.

### 2.2 Tipos de datos

Cada columna tiene un tipo: número entero (`int64`), decimal (`float64`), texto (`object`/`str`), fecha (`datetime64`) o verdadero/falso (`bool`). Un tipo incorrecto (por ejemplo, un importe leído como texto) impediría sumar o promediar. Revisamos la tabla principal:

In [4]:
ventas.dtypes.to_frame("tipo").T

,order_id,order_date,order_time,order_status,sales_channel,customer_id,customer_name,customer_age,gender,customer_segment,customer_type,customer_city,customer_state,customer_country,region,customer_postal_code,payment_method,payment_status,currency,shipping_method,warehouse,delivery_days,estimated_delivery_days,delivery_status,return_status,return_reason,customer_rating,review_sentiment,customer_review,marketing_channel,campaign_name,coupon_code,loyalty_points_earned,loyalty_points_redeemed,quantity,gross_sales,discount_amount,tax_amount,shipping_cost,net_sales,product_cost,profit,profit_margin_percentage,customer_lifetime_value,is_repeat_customer,customer_order_count
tipo,str,datetime64[us],str,str,str,str,str,int64,str,str,str,str,str,str,str,int64,str,str,str,str,str,float64,float64,str,str,str,float64,str,str,str,str,str,int64,int64,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,bool,int64


Los importes son numéricos y `order_date` ya es fecha. `order_time` se ha leído como texto (`HH:MM:SS`); no es un problema, en la sección 3 extraeremos de ella la hora.

### 2.3 Filas duplicadas y claves únicas

- **Fila duplicada:** dos filas idénticas en todas sus columnas. Inflarían sumas y recuentos.
- **Clave primaria:** columna que debe identificar cada fila de forma única (`order_id` en ventas, `customer_id` en clientes, `product_id` en productos). Si se repite, al cruzar tablas se multiplicarían filas.
- En `items` la clave es la **combinación** `order_id` + `product_id` (un pedido tiene varios productos).

In [5]:
claves = {"ventas": ["order_id"], "items": ["order_id", "product_id"],
          "clientes": ["customer_id"], "productos": ["product_id"]}

for nombre, df in tablas.items():
    print(f"{nombre:<10} filas duplicadas: {df.duplicated().sum():>3} | "
          f"clave {claves[nombre]} repetida: {df.duplicated(subset=claves[nombre]).sum()}")

ventas     filas duplicadas:   0 | clave ['order_id'] repetida: 0
items      filas duplicadas:   0 | clave ['order_id', 'product_id'] repetida: 87
clientes   filas duplicadas:   0 | clave ['customer_id'] repetida: 0
productos  filas duplicadas:   0 | clave ['product_id'] repetida: 0


**Resultado:** no hay duplicados y todas las claves son únicas. No hace falta eliminar filas.

### 2.4 Integridad referencial: ¿las tablas se pueden cruzar?

Las tablas se relacionan a través de sus claves:

```
clientes.customer_id  ──<  ventas.customer_id
ventas.order_id       ──<  items.order_id
productos.product_id  ──<  items.product_id
```

*Integridad referencial* significa que cada clave que aparece en una tabla existe en la tabla a la que apunta. Por ejemplo, que no haya ventas a un cliente que no está en el maestro de clientes. Si la hubiera, en Power BI aparecerían filas "en blanco" al filtrar.

In [6]:
comprobaciones = {
    "ventas cuyo cliente no existe en clientes":    ~ventas["customer_id"].isin(clientes["customer_id"]),
    "items cuyo pedido no existe en ventas":        ~items["order_id"].isin(ventas["order_id"]),
    "items cuyo producto no existe en productos":   ~items["product_id"].isin(productos["product_id"]),
    "pedidos sin ninguna línea en items":           ~ventas["order_id"].isin(items["order_id"]),
    "clientes sin ningún pedido":                   ~clientes["customer_id"].isin(ventas["customer_id"]),
    "productos que nunca se han vendido":           ~productos["product_id"].isin(items["product_id"]),
}
for texto, mascara in comprobaciones.items():
    print(f"{texto:<45} {mascara.sum():>5}")

ventas cuyo cliente no existe en clientes         0
items cuyo pedido no existe en ventas             0
items cuyo producto no existe en productos        0
pedidos sin ninguna línea en items                0
clientes sin ningún pedido                       89
productos que nunca se han vendido                0


**Resultado:** todas las relaciones son correctas. Las únicas "huérfanas" son **clientes sin pedidos** (registrados pero que nunca compraron). No es un error: se conservan y se marcarán con una columna `tiene_pedidos` para poder filtrarlos.

### 2.5 Valores nulos (vacíos)

Un nulo (`NaN`) es una celda vacía. Antes de decidir qué hacer con ellos hay que entender **por qué** están vacíos: no es lo mismo un dato que se perdió que un dato que no puede existir.

In [7]:
for nombre, df in tablas.items():
    nulos = df.isna().sum()
    nulos = nulos[nulos > 0]
    print(f"=== {nombre} ===")
    print(nulos.to_string() if len(nulos) else "sin nulos")
    print()

=== ventas ===
delivery_days               24557
estimated_delivery_days     24557
return_status              128654
return_reason              128654
customer_rating             24557
review_sentiment            24557
customer_review             24557
campaign_name               83233
coupon_code                110502

=== items ===
sin nulos

=== clientes ===
sin nulos

=== productos ===
sin nulos



Solo `ventas` tiene nulos, y se repiten cantidades exactas (24.557, 128.654...). Eso sugiere que no son errores al azar sino que dependen de algo. Cruzamos los nulos con el estado del pedido (`order_status`):

In [8]:
columnas_con_nulos = ["delivery_days", "customer_rating", "return_status", "coupon_code", "campaign_name"]

# Para cada estado del pedido: cuántas filas tienen el dato vacío
ventas[columnas_con_nulos].isna().groupby(ventas["order_status"]).sum() \
    .assign(total_pedidos=ventas["order_status"].value_counts())

,delivery_days,customer_rating,return_status,coupon_code,campaign_name,total_pedidos
order_status,,,,,,
Cancelled,8398,8398,8398,6691,5102,8398
Completed,0,0,113559,90876,68254,113559
Pending,6697,6697,6697,5393,4154,6697
Returned,9462,9462,0,7542,5723,9462


**Interpretación:**

| Columnas | Nulos | Por qué están vacías | Decisión (sección 3) |
|---|---|---|---|
| `delivery_days`, `estimated_delivery_days`, `customer_rating`, `review_sentiment`, `customer_review` | 24.557 | Son **todos** los pedidos *Cancelled*, *Returned* y *Pending*: no se entregaron o no se valoraron | Se **mantienen vacíos**. Rellenarlos con 0 falsearía las medias (un 0 en valoración bajaría la nota media) |
| `return_status`, `return_reason` | 128.654 | Solo se rellenan cuando el pedido se devolvió | Texto: `"Not Returned"` / `"Sin devolución"` |
| `coupon_code` | 110.502 | El pedido no usó cupón | Texto: `"Sin cupón"` |
| `campaign_name` | 83.233 | El pedido no vino de ninguna campaña | Texto: `"Sin campaña"` |

Regla general: **los nulos de texto se sustituyen por una etiqueta** (así aparecen como categoría en los gráficos), y **los nulos numéricos se dejan vacíos** (Power BI los ignora al promediar, que es lo correcto).

### 2.6 Coherencia de los importes

Comprobamos que las columnas de dinero cumplen las fórmulas que cabría esperar. Si se cumplen, podemos confiar en ellas y sabemos qué significa cada una:

- `net_sales = gross_sales - discount_amount + tax_amount + shipping_cost`
- `profit = net_sales - product_cost - shipping_cost`
- `profit_margin_percentage = profit / net_sales x 100`

Se admite una diferencia de hasta 0,05 por redondeos.

In [9]:
def cumple(calculado, real, tolerancia=0.05):
    "Porcentaje de filas en que el valor calculado coincide con el real."
    return f"{((calculado - real).abs() <= tolerancia).mean():.2%}"

for nombre, df in {"ventas": ventas, "items": items}.items():
    print(f"=== {nombre} ===")
    print("  net_sales :", cumple(df["gross_sales"] - df["discount_amount"] + df["tax_amount"] + df["shipping_cost"], df["net_sales"]))
    print("  profit    :", cumple(df["net_sales"] - df["product_cost"] - df["shipping_cost"], df["profit"]))

print("ventas margen %  :", cumple(ventas["profit"] / ventas["net_sales"] * 100, ventas["profit_margin_percentage"]))
print("items bruto      :", cumple(items["unit_price"] * items["quantity"], items["gross_sales"]))
print("items descuento  :", cumple(items["discount_percentage"] * items["gross_sales"], items["discount_amount"]))

=== ventas ===
  net_sales : 100.00%
  profit    : 100.00%
=== items ===
  net_sales : 100.00%
  profit    : 100.00%
ventas margen %  : 100.00%
items bruto      : 100.00%
items descuento  : 100.00%


**Resultado:** todas las fórmulas se cumplen en el 100% de las filas. Además vemos que `discount_percentage` está en formato **fracción** (0,34 = 34%), algo a tener en cuenta al mostrarlo en Power BI.

#### ¿Cuadra la cabecera del pedido con la suma de sus líneas?

Cada pedido de `ventas` debería tener el mismo importe que la suma de sus productos en `items`.

In [10]:
suma_items = items.groupby("order_id")[["quantity", "net_sales", "profit"]].sum()

cruce = ventas.set_index("order_id")[["quantity", "net_sales", "profit", "order_status"]] \
              .join(suma_items, rsuffix="_items")

cruce["cuadra"] = (cruce["net_sales"] - cruce["net_sales_items"]).abs() <= 0.05

print(f"Pedidos que NO cuadran: {(~cruce['cuadra']).sum():,} de {len(cruce):,} "
      f"({(~cruce['cuadra']).mean():.1%})")
display(cruce.loc[~cruce["cuadra"]].head(5))
cruce.groupby("order_status")["cuadra"].apply(lambda s: f"{(~s).mean():.1%} no cuadran")

Pedidos que NO cuadran: 11,226 de 138,116 (8.1%)


C:\Users\Lucas\AppData\Local\Temp\ipykernel_22100\4227681990.py:4: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  .join(suma_items, rsuffix="_items")


,quantity,net_sales,profit,order_status,quantity_items,net_sales_items,profit_items,cuadra
order_id,,,,,,,,
ORD-163733,7,486.48,310.33,Completed,13,"1,579.70",965.04,False
ORD-652783,14,"4,497.43",878.48,Completed,24,"6,419.61","1,681.45",False
ORD-537228,1,144.13,88.36,Completed,8,567.43,276.35,False
ORD-697815,8,"4,286.91","1,615.04",Completed,11,"4,396.29","1,633.77",False
ORD-509719,4,"1,478.86",789.55,Completed,10,"2,665.42","1,317.27",False


order_status
Cancelled    7.8% no cuadran
Completed    8.2% no cuadran
Pending      7.9% no cuadran
Returned     8.2% no cuadran
Name: cuadra, dtype: str

**Hallazgo importante:** en torno al **8% de los pedidos** la cabecera no coincide con la suma de sus líneas (ni en importe ni en cantidad), y ocurre en todos los estados por igual. Es una inconsistencia de la fuente que no podemos corregir porque no sabemos cuál de las dos versiones es la buena.

**Decisión:**

- `ventas` es la **fuente oficial para cifras por pedido** (facturación, beneficio, ticket medio, devoluciones...). Es la que coincide con las estadísticas oficiales (lo verificamos en la sección 4).
- `items` se usa **solo para analizar productos** (categorías, marcas, proveedores...).
- **No se mezclan** las dos tablas en un mismo indicador.
- Se añade a `ventas` la columna `cuadra_con_items` (Verdadero/Falso) para poder identificar o excluir estos pedidos si hace falta.

### 2.7 Datos de cliente repetidos en `ventas`

La tabla `ventas` incluye nombre, edad, ciudad, etc. del cliente en cada pedido, y `clientes` contiene esos mismos datos. Si coinciden, guardarlos en `ventas` es redundante: ocupa espacio y crea el riesgo de que un mismo cliente tenga dos versiones distintas de un dato.

Comparamos, para cada columna común a las dos tablas, si el valor es idéntico.

In [11]:
columnas_comunes = [c for c in clientes.columns if c in ventas.columns and c != "customer_id"]

comparacion = ventas[["customer_id"] + columnas_comunes].merge(
    clientes, on="customer_id", suffixes=("_ventas", "_maestro"))

diferencias = {col: (comparacion[f"{col}_ventas"] != comparacion[f"{col}_maestro"]).sum()
               for col in columnas_comunes}
pd.Series(diferencias, name="filas distintas").to_frame()

,filas distintas
customer_name,0
customer_age,0
gender,0
customer_segment,0
customer_city,0
customer_state,0
customer_country,0
region,0
customer_postal_code,0


Todas coinciden. Además, `ventas` tiene 4 columnas que describen **al cliente** y no al pedido (`customer_type`, `customer_lifetime_value`, `is_repeat_customer`, `customer_order_count`). Comprobamos que cada cliente tiene un único valor en ellas:

In [12]:
atributos_cliente = ["customer_type", "customer_lifetime_value", "is_repeat_customer", "customer_order_count"]

(ventas.groupby("customer_id")[atributos_cliente].nunique() > 1).sum() \
    .to_frame("clientes con más de un valor")

,clientes con más de un valor
customer_type,0
customer_lifetime_value,0
is_repeat_customer,0
customer_order_count,0


**Resultado:** cada cliente tiene un solo valor. **Decisión:** en la sección 3 estas 4 columnas se trasladan a `clientes` y todas las columnas de cliente se eliminan de `ventas`, que se queda solo con `customer_id` para relacionarse.

> Nota: `customer_order_count` viene calculado por la fuente y no siempre coincide con los pedidos que hay en este dataset (que es una muestra). Se conserva, pero para contar pedidos es mejor contar filas de `ventas`.

### 2.8 Moneda

Hay pedidos en 7 monedas (`currency`). Si los importes estuvieran en moneda local, sumarlos directamente sería un error (no se pueden sumar dólares y rupias). Comparamos el importe medio por pedido en cada moneda:

In [13]:
ventas.groupby("currency")["net_sales"].agg(["count", "mean"]).sort_values("mean")

,count,mean
currency,,
USD,82600,"1,237.65"
AED,4172,"1,238.25"
AUD,6794,"1,297.28"
CAD,7015,"1,311.31"
INR,5707,"1,352.42"
GBP,20235,"1,381.92"
EUR,11593,"1,383.95"


**Interpretación:** la media es prácticamente la misma en todas las monedas (unos 1.200-1.400). Si los importes en INR fueran rupias reales, serían unas 80 veces mayores que los de USD. Por tanto, **todos los importes están expresados en la misma unidad** (probablemente USD) y `currency` solo indica la moneda del país del cliente. **No se convierte nada**; se deja documentado como supuesto.

### 2.9 Rangos de valores

Revisamos mínimos y máximos de las columnas numéricas para detectar valores imposibles (edades negativas, valoraciones fuera de 1-5, etc.).

In [14]:
ventas[["customer_age", "quantity", "net_sales", "profit", "profit_margin_percentage",
        "delivery_days", "customer_rating"]].describe().T[["min", "mean", "max"]]

,min,mean,max
customer_age,18.00,45.86,74.00
quantity,1.00,5.62,24.00
net_sales,5.80,"1,282.50","16,218.00"
profit,"-1,281.59",551.32,"7,083.68"
profit_margin_percentage,-59.55,45.07,77.59
delivery_days,0.00,4.56,14.00
customer_rating,1.60,3.68,5.00


Los rangos son razonables: edades de 18 a 74, valoraciones de 1,6 a 5, entregas de 0 a 14 días. El **beneficio negativo** (pérdidas) es posible cuando el descuento es grande, así que no es un error y se mantiene.

**Fin de la exploración.** Resumen de decisiones para la transformación:

1. Rellenar nulos de texto con etiquetas; mantener vacíos los numéricos.
2. Mover los atributos de cliente de `ventas` a `clientes`.
3. Marcar los pedidos cuya cabecera no cuadra con sus líneas.
4. Redondear decimales y añadir columnas útiles para el análisis (año, mes, rango de edad...).
5. Construir una tabla calendario.

## 3. Transformación

### El modelo en estrella

En lugar de juntar todo en una tabla enorme, organizamos los datos en el formato que mejor funciona en Power BI: el **modelo en estrella**. Tiene dos tipos de tablas:

- **Tablas de hechos (`fact_`)**: registran *sucesos* con importes (pedidos, líneas de pedido). Son largas y tienen las cifras que se suman.
- **Tablas de dimensión (`dim_`)**: describen *quién, qué y cuándo* (clientes, productos, fechas). Son cortas y se usan para filtrar y agrupar.

```
                 dim_clientes          dim_fecha
                   │      │             │      │
                   ▼      ▼             ▼      ▼
             fact_ventas        fact_items ◄── dim_productos
            (1 fila/pedido)   (1 fila/producto en pedido)
```

Cada tabla de hechos se conecta directamente a las dimensiones que necesita. Las dos tablas de hechos **no se conectan entre sí**, de acuerdo con la decisión de 2.6. Para que `fact_items` pueda filtrarse por fecha y cliente, le copiamos `order_date`, `customer_id` y `order_status` desde `ventas`.

Trabajamos sobre **copias** (`.copy()`) para no alterar las tablas originales cargadas en memoria.

### 3.1 `dim_clientes`: tabla de clientes

1. Partimos del maestro de clientes.
2. Añadimos los 4 atributos de cliente que estaban en `ventas` (2.7). Usamos `drop_duplicates` para quedarnos con una fila por cliente antes de unir.
3. Los clientes que nunca compraron no tienen esos atributos: `customer_type` se rellena con `"Sin pedidos"` y `is_repeat_customer` con Falso; las métricas numéricas se dejan vacías.
4. Añadimos `tiene_pedidos` y `rango_edad` (agrupar edades facilita los gráficos).

In [15]:
dim_clientes = clientes.copy()

atributos = ventas[["customer_id"] + atributos_cliente].drop_duplicates("customer_id")
dim_clientes = dim_clientes.merge(atributos, on="customer_id", how="left")   # 'left' conserva todos los clientes

dim_clientes["tiene_pedidos"]      = dim_clientes["customer_id"].isin(ventas["customer_id"])
dim_clientes["customer_type"]      = dim_clientes["customer_type"].fillna("Sin pedidos")
dim_clientes["is_repeat_customer"] = dim_clientes["is_repeat_customer"].fillna(False).astype(bool)

# pd.cut reparte las edades en intervalos: (17,25] -> "18-25", (25,35] -> "26-35", ...
dim_clientes["rango_edad"] = pd.cut(dim_clientes["customer_age"],
                                    bins=[17, 25, 35, 45, 55, 65, 75],
                                    labels=["18-25", "26-35", "36-45", "46-55", "56-65", "66-74"])

print(dim_clientes.shape)
dim_clientes.head(3)

(25000, 17)


,customer_id,customer_name,customer_age,gender,customer_segment,customer_city,customer_state,customer_country,region,customer_postal_code,customer_acquisition_cost,customer_type,customer_lifetime_value,is_repeat_customer,customer_order_count,tiene_pedidos,rango_edad
0,CUST-000001,Donna Miller,54,Female,Consumer,Laurenland,Dubai,UAE,South,11253,13.58,Loyal,"11,125.84",True,10.00,True,46-55
1,CUST-000002,Joseph James,61,Male,Consumer,Lake Peter,Lower Saxony,Germany,North,47778,27.45,Loyal,"18,138.96",True,10.00,True,56-65
2,CUST-000003,Daniel Smith,47,Male,Consumer,Seanview,North Rhine,Germany,West,13900,31.53,Loyal,"7,798.59",True,7.00,True,46-55


### 3.2 `dim_productos`: tabla de productos

El catálogo ya está limpio. Solo añadimos el **margen unitario** (precio - coste) en importe y en porcentaje, útil para comparar la rentabilidad de categorías y marcas.

In [16]:
dim_productos = productos.copy()

dim_productos["margen_unitario"]     = dim_productos["unit_price"] - dim_productos["product_cost"]
dim_productos["margen_unitario_pct"] = dim_productos["margen_unitario"] / dim_productos["unit_price"] * 100

print(dim_productos.shape)
dim_productos.head(3)

(1175, 11)


,product_id,product_name,product_category,product_subcategory,brand,supplier,unit_price,product_cost,product_rating,margen_unitario,margen_unitario_pct
0,PROD-000001,Apple Sharable bifurcated algorithm Smartphones,Electronics,Smartphones,Apple,Alibaba,686.90,474.06,4.30,212.84,30.99
1,PROD-000002,Google User-centric even-keeled encryption Sma...,Electronics,Smartphones,Google,Euro Logistics,978.26,519.66,2.90,458.60,46.88
2,PROD-000003,LG Face-to-face client-driven support Smartphones,Electronics,Smartphones,LG,Asian Manufacturing,275.51,185.48,3.90,90.03,32.68


### 3.3 `fact_ventas`: tabla de pedidos

1. **Quitamos** las columnas de cliente (ya están en `dim_clientes`) y nos quedamos con `customer_id` como enlace.
2. **Rellenamos** los nulos de texto según la tabla de 2.5.
3. **Añadimos** columnas de tiempo derivadas de la fecha y la hora, y la marca `cuadra_con_items` (2.6).

In [17]:
fact_ventas = ventas.copy()

# 1. Columnas de cliente que pasan a dim_clientes
columnas_a_quitar = columnas_comunes + atributos_cliente
fact_ventas = fact_ventas.drop(columns=columnas_a_quitar)
print("Columnas eliminadas:", columnas_a_quitar)

# 2. Nulos de texto -> etiqueta (los numéricos se quedan vacíos a propósito)
fact_ventas = fact_ventas.fillna({
    "return_status": "Not Returned",
    "return_reason": "Sin devolución",
    "coupon_code":   "Sin cupón",
    "campaign_name": "Sin campaña",
})
fact_ventas["tiene_cupon"] = fact_ventas["coupon_code"] != "Sin cupón"

# 3. Columnas derivadas
fact_ventas["hora"] = pd.to_datetime(fact_ventas["order_time"], format="%H:%M:%S").dt.hour
fact_ventas["cuadra_con_items"] = fact_ventas["order_id"].map(cruce["cuadra"])

print(fact_ventas.shape)
fact_ventas.head(3)

Columnas eliminadas: ['customer_name', 'customer_age', 'gender', 'customer_segment', 'customer_city', 'customer_state', 'customer_country', 'region', 'customer_postal_code', 'customer_type', 'customer_lifetime_value', 'is_repeat_customer', 'customer_order_count']
(138116, 36)


,order_id,order_date,order_time,order_status,sales_channel,customer_id,payment_method,payment_status,currency,shipping_method,warehouse,delivery_days,estimated_delivery_days,delivery_status,return_status,return_reason,customer_rating,review_sentiment,customer_review,marketing_channel,campaign_name,coupon_code,loyalty_points_earned,loyalty_points_redeemed,quantity,gross_sales,discount_amount,tax_amount,shipping_cost,net_sales,product_cost,profit,profit_margin_percentage,tiene_cupon,hora,cuadra_con_items
0,ORD-301242,2023-11-06,16:37:47,Completed,Mobile App,CUST-003102,Digital Wallet,Paid,USD,Standard,WH-003,3.00,3.00,On Time,Not Returned,Sin devolución,3.50,Positive,Satisfied with the purchase.,Direct,Default_Campaign,Sin cupón,94,44,5,"1,350.19",477.89,61.07,12.03,945.40,588.33,345.04,36.50,False,16,True
1,ORD-773460,2025-12-24,01:22:36,Completed,Website,CUST-003124,Debit Card,Pending,EUR,Economy,WH-005,10.00,10.00,On Time,Not Returned,Sin devolución,3.60,Positive,Good value for money.,Direct,Default_Campaign,Sin cupón,201,171,8,"3,144.64","1,456.06",320.83,9.00,"2,018.41","2,031.88",-22.47,-1.11,False,1,True
2,ORD-449374,2021-07-05,14:24:21,Completed,Mobile App,CUST-012496,Cash on Delivery,Paid,USD,Express,WH-015,3.00,3.00,On Time,Not Returned,Sin devolución,4.30,Positive,Good product. Works as expected.,YouTube,Sin campaña,Sin cupón,46,16,5,522.81,97.30,29.79,13.05,468.35,257.73,197.57,42.18,False,14,True


> El año, el mes y el día de la semana **no** se añaden aquí: estarán en `dim_fecha` (3.5), que se relaciona con `order_date`. Así se definen una sola vez para las dos tablas de hechos.

### 3.4 `fact_items`: tabla de líneas de pedido

Añadimos desde `ventas` la fecha, el cliente y el estado del pedido. Gracias a eso, en Power BI esta tabla podrá filtrarse por fecha y por cliente directamente, y se podrán excluir las líneas de pedidos cancelados.

Pasamos `discount_percentage` de fracción (0,34) a porcentaje (34) para que sea coherente con `profit_margin_percentage`.

In [18]:
fact_items = items.merge(ventas[["order_id", "order_date", "customer_id", "order_status"]],
                         on="order_id", how="left")

fact_items["discount_percentage"] = fact_items["discount_percentage"] * 100

print(fact_items.shape)
fact_items.head(3)

(397569, 15)


,order_id,product_id,quantity,unit_price,discount_percentage,discount_amount,gross_sales,tax_amount,shipping_cost,net_sales,product_cost,profit,order_date,customer_id,order_status
0,ORD-301242,PROD-000738,2,244.40,33.96,166.00,488.80,22.60,6.48,351.88,173.28,172.12,2023-11-06,CUST-003102,Completed
1,ORD-301242,PROD-000181,3,287.13,36.21,311.89,861.39,38.47,5.55,593.52,415.05,172.92,2023-11-06,CUST-003102,Completed
2,ORD-773460,PROD-000973,4,754.80,46.83,"1,413.89","3,019.20",305.01,9.00,"1,919.32","1,976.16",-65.84,2025-12-24,CUST-003124,Completed


### 3.5 `dim_fecha`: tabla calendario

Una tabla con **un día por fila** que cubre todo el periodo de los datos. Power BI la necesita para agrupar por año, trimestre, mes o día de la semana, y para funciones de tiempo (comparar con el año anterior, acumulados...).

Los nombres de mes y día se escriben en español con listas propias, porque los nombres automáticos dependerían del idioma del ordenador. Incluimos también columnas numéricas (`mes_num`, `dia_semana_num`) para que Power BI pueda **ordenar** los meses cronológicamente y no alfabéticamente.

In [19]:
MESES = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio", "Julio",
         "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre"]
DIAS  = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]

fechas = pd.date_range(fact_ventas["order_date"].min(), fact_ventas["order_date"].max(), freq="D")

dim_fecha = pd.DataFrame({"fecha": fechas})
dim_fecha["anio"]           = dim_fecha["fecha"].dt.year
dim_fecha["trimestre"]      = "T" + dim_fecha["fecha"].dt.quarter.astype(str)
dim_fecha["mes_num"]        = dim_fecha["fecha"].dt.month
dim_fecha["mes"]            = dim_fecha["mes_num"].map(lambda m: MESES[m - 1])
dim_fecha["anio_mes"]       = dim_fecha["fecha"].dt.strftime("%Y-%m")
dim_fecha["dia_semana_num"] = dim_fecha["fecha"].dt.dayofweek + 1          # 1 = lunes
dim_fecha["dia_semana"]     = dim_fecha["dia_semana_num"].map(lambda d: DIAS[d - 1])
dim_fecha["fin_de_semana"]  = dim_fecha["dia_semana_num"] >= 6

print(dim_fecha.shape, "| desde", dim_fecha["fecha"].min().date(), "hasta", dim_fecha["fecha"].max().date())
dim_fecha.head(3)

(1826, 9) | desde 2021-01-01 hasta 2025-12-31


C:\Users\Lucas\AppData\Local\Temp\ipykernel_22100\172040355.py:13: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  dim_fecha["dia_semana_num"] = dim_fecha["fecha"].dt.dayofweek + 1          # 1 = lunes


,fecha,anio,trimestre,mes_num,mes,anio_mes,dia_semana_num,dia_semana,fin_de_semana
0,2021-01-01,2021,T1,1,Enero,2021-01,5,Viernes,False
1,2021-01-02,2021,T1,1,Enero,2021-01,6,Sábado,True
2,2021-01-03,2021,T1,1,Enero,2021-01,7,Domingo,True


### 3.6 Redondeo de decimales

Algunos importes arrastran errores de coma flotante (por ejemplo `12.030000000000001` en lugar de `12.03`). Es un efecto de cómo los ordenadores guardan los decimales. Redondeamos todas las columnas decimales a 2 cifras: el CSV queda más limpio y las cifras se leen mejor.

In [20]:
modelo = {"fact_ventas": fact_ventas, "fact_items": fact_items, "dim_clientes": dim_clientes,
          "dim_productos": dim_productos, "dim_fecha": dim_fecha}

for nombre, df in modelo.items():
    decimales = df.select_dtypes("float").columns
    df[decimales] = df[decimales].round(2)

fact_ventas[["gross_sales", "shipping_cost", "net_sales", "profit"]].head(3)

,gross_sales,shipping_cost,net_sales,profit
0,"1,350.19",12.03,945.40,345.04
1,"3,144.64",9.00,"2,018.41",-22.47
2,522.81,13.05,468.35,197.57


## 4. Validación final

Comparamos nuestros resultados con `dataset_statistics.csv`, que contiene los totales oficiales del dataset. Si coinciden, sabemos que la limpieza **no ha perdido ni alterado datos**.

Las cifras oficiales vienen como texto con símbolos (`"$177,134,263.74"`, `"6.85%"`), así que primero les quitamos `$`, `,` y `%` para convertirlas en números.

> `Total Products Used` (138.116) es igual al número de pedidos, no al de productos distintos, así que no se compara.

In [21]:
oficial = estadisticas.iloc[0]

def a_numero(texto):
    return float(str(texto).replace("$", "").replace(",", "").replace("%", ""))

n_pedidos = len(fact_ventas)
validacion = pd.DataFrame([
    ("Total de pedidos",      a_numero(oficial["Total Transactions"]), n_pedidos),
    ("Clientes con pedidos",  a_numero(oficial["Total Customers"]),    fact_ventas["customer_id"].nunique()),
    ("Facturación total",     a_numero(oficial["Total Revenue"]),      fact_ventas["net_sales"].sum()),
    ("Beneficio total",       a_numero(oficial["Total Profit"]),       fact_ventas["profit"].sum()),
    ("Ticket medio",          a_numero(oficial["Average Order Value"]), fact_ventas["net_sales"].mean()),
    ("Valoración media",      a_numero(oficial["Average Rating"]),     fact_ventas["customer_rating"].mean()),
    ("Tasa de devolución %",  a_numero(oficial["Return Rate"]),        (fact_ventas["order_status"] == "Returned").mean() * 100),
    ("Tasa de cancelación %", a_numero(oficial["Cancellation Rate"]),  (fact_ventas["order_status"] == "Cancelled").mean() * 100),
], columns=["Métrica", "Oficial", "Calculado"])

# Las cifras oficiales están redondeadas a 2 decimales: admitimos una diferencia
# de 0,01 o de una parte entre 10.000 (lo que sea mayor)
validacion["Coincide"] = np.isclose(validacion["Oficial"], validacion["Calculado"], rtol=1e-4, atol=0.01)
validacion

,Métrica,Oficial,Calculado,Coincide
0,Total de pedidos,"138,116.00","138,116.00",True
1,Clientes con pedidos,"24,911.00","24,911.00",True
2,Facturación total,"177,134,263.74","177,134,263.74",True
3,Beneficio total,"76,146,395.76","76,146,395.76",True
4,Ticket medio,"1,282.50","1,282.50",True
5,Valoración media,3.68,3.68,True
6,Tasa de devolución %,6.85,6.85,True
7,Tasa de cancelación %,6.08,6.08,True


Si todas las filas muestran `True` en **Coincide**, las tablas limpias reproducen exactamente las cifras oficiales. Esto confirma además que `fact_ventas` es la tabla correcta para los indicadores por pedido (decisión de 2.6).

Como última comprobación, verificamos que ninguna transformación ha añadido ni perdido filas:

In [22]:
assert len(fact_ventas)  == len(ventas),   "fact_ventas no tiene las mismas filas que ventas"
assert len(fact_items)   == len(items),    "fact_items no tiene las mismas filas que items"
assert len(dim_clientes) == len(clientes), "dim_clientes no tiene las mismas filas que clientes"
assert dim_clientes["customer_id"].is_unique and dim_productos["product_id"].is_unique
assert validacion["Coincide"].all(), "Alguna métrica no coincide con las estadísticas oficiales"
print("Todas las comprobaciones son correctas ✔")

Todas las comprobaciones son correctas ✔


## 5. Carga: exportación de las tablas limpias

Guardamos cada tabla en un CSV dentro de la carpeta `Datos_limpios` (se crea si no existe):

- `index=False`: no se guarda el número de fila interno de pandas.
- `encoding="utf-8-sig"`: conserva tildes y caracteres especiales (`Baden-Württemberg`, `Miércoles`) y hace que Excel y Power BI los lean bien.
- Separador de columnas: coma. Separador decimal: **punto**.

> **Importante para Power BI:** al importar, las columnas con decimales deben configurarse con la configuración regional **Inglés (EE. UU.)**. Con configuración española, Power BI interpreta el punto como separador de miles y multiplica los valores.

In [23]:
RUTA_SALIDA = Path("Datos_limpios")
RUTA_SALIDA.mkdir(exist_ok=True)

for nombre, df in modelo.items():
    df.to_csv(RUTA_SALIDA / f"{nombre}.csv", index=False, encoding="utf-8-sig")
    print(f"{nombre + '.csv':<20} {df.shape[0]:>8,} filas x {df.shape[1]:>2} columnas")

fact_ventas.csv       138,116 filas x 36 columnas
fact_items.csv        397,569 filas x 15 columnas
dim_clientes.csv       25,000 filas x 17 columnas
dim_productos.csv       1,175 filas x 11 columnas
dim_fecha.csv           1,826 filas x  9 columnas
